In [161]:
import pandas as pd
from datetime import datetime

import sys
from pathlib import Path

In [166]:
sys.path.append(r"C:/Users/sanya/structured-products-analytics")

from src.reverse_convertible import ReverseConvertible
from src.scenario_engine import ScenarioEngine


In [156]:
portfolio = pd.DataFrame({
    "product_id": [
        "CH1483491150",
        "CH1449111066",
        "CH1461018793"
    ],
    
    "product_type": [
        "BRC",
        "MBRC",
        "MBRC"
    ],
    
    "type_style": [
        "European",
        "European",
        "European"
    ],
    
    "underlyings": [
        ["ALCON"],
        ["ABB", "HOLCIM", "NOVARTIS", "ROCHE"],
        ["ABB", "LONZA", "NESTLE"]
    ],
    
    "underlying_isins": [
        ["CH0432492467"],
        [
            "CH0012221716",
            "CH0012214059",
            "CH0012005267",
            "CH0012032048"
        ],
        [
            "CH0012221716",
            "CH0013841017",
            "CH0038863350"
        ]
    ],
    
    "currency": [
        "CHF",
        "CHF",
        "CHF"
    ],
    
    "position_units": [
        10,
        5,
        1
    ],
    
    "notional": [
        1000,
        1000,
        10000
    ],
    
    "cost_price": [
        1.00,
        0.98,
        1.00
    ],
    
    "initial_levels": [
        [59.72],
        [35.00, 70.00, 90.00, 250.00],
        [53.94, 555.20, 72.49]
    ],
    
    "current_spots": [
        [58.76],
        [34.00, 68.00, 92.00, 245.00],
        [53.94, 555.20, 72.49]   # replace with live levels
    ],
    
    
    "strike": [
        [59.72],
        [35.00, 70.00, 90.00, 250.00],
        [53.94, 555.20, 72.49]
    ],
    
    "barrier_pct": [
        0.70,
        0.70,
        0.70
    ],
    
    "coupon": [
        0.04,
        0.0675,
        0.0866
    ],
    
    "initial_fixing_date": [
        "2025-11-10",
        "2025-12-30",
        "2025-08-19"
    ],
    
    
    "maturity_date": [
        "2026-11-17",
        "2026-12-28",
        "2026-08-19"
    ],
    
    "barrier_breached": [
        False,
        True,
        False
    ]
})

In [77]:
#portfolio.to_csv("data/raw/portfolio.csv", index=False)
portfolio

,product_id,product_type,type_style,underlyings,underlying_isins,currency,position_units,notional,cost_price,initial_levels,current_spots,strike,barrier_pct,coupon,initial_fixing_date,maturity_date,barrier_breached
0,CH1483491150,BRC,European,[ALCON],[CH0432492467],CHF,10,1000,1.00,[59.72],[58.76],[59.72],0.7,0.0400,2025-11-10,2026-11-17,False
1,CH1449111066,MBRC,European,"[ABB, HOLCIM, NOVARTIS, ROCHE]","[CH0012221716, CH0012214059, CH0012005267, CH0...",CHF,5,1000,0.98,"[35.0, 70.0, 90.0, 250.0]","[34.0, 68.0, 92.0, 245.0]","[35.0, 70.0, 90.0, 250.0]",0.7,0.0675,2025-12-30,2026-12-28,True
2,CH1461018793,MBRC,European,"[ABB, LONZA, NESTLE]","[CH0012221716, CH0013841017, CH0038863350]",CHF,1,10000,1.00,"[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]",0.7,0.0866,2025-08-19,2026-08-19,False


In [97]:
rc = ReverseConvertible(portfolio.iloc[1], [-10, 5, 0, -3])
print(rc.final_levels)

[30.6, 71.4, 92.0, 237.65]


In [179]:
import pandas as pd
import numpy as np
from datetime import datetime
from src.reverse_convertible import ReverseConvertible


class PortfolioAnalytics:

    def __init__(self, portfolio: pd.DataFrame):
        self.portfolio = portfolio
        self.product_df = None

    # =========================
    # 1. PRODUCT LEVEL
    # =========================
    def build_product_analytics(self) -> pd.DataFrame:
        rows = []

        for _, row in self.portfolio.iterrows():
            rc = ReverseConvertible(row)
            s = rc.summary()

            rows.append({
                "product_id": row["product_id"],
                "product_type": row["product_type"],
                "type_style": row["type_style"],
                "currency": row["currency"],
                "position_units": row["position_units"],
                "notional": row["notional"],
                "coupon": row["coupon"],
                "underlyings": ", ".join(row["underlyings"]),
                "n_underlyings": len(row["underlyings"]),
                "initial_fixing_date": row["initial_fixing_date"],
                "maturity_date": row["maturity_date"],

                "performance": s["performance"],
                "barrier_breached": s["barrier_breached"],
                "worst_underlying": s["worst_underlying"],
                "payoff_per_unit": s["payoff_per_unit"],
                "total_payoff": s["total_payoff"],
                "total_cost": s["total_cost"],
                "pnl": s["pnl"],
                "return_pct": s["return_pct"],
                "return_pa": s["return_pa"],
                "distance_to_barrier": s["distance_to_barrier"],
                "break_even": s["break_even"]
            })

        self.product_df = pd.DataFrame(rows)
        return self.product_df

    # =========================
    # 2. PORTFOLIO SUMMARY
    # =========================
    def portfolio_summary(self) -> pd.DataFrame:
        df = self._require_product_df()

        total_cost = df["total_cost"].sum()
        total_pnl = df["pnl"].sum()

        return pd.DataFrame([{
            "n_products": len(df),
            "n_brc": (df["product_type"] == "BRC").sum(),
            "n_mbrc": (df["product_type"] == "MBRC").sum(),
            "total_cost": total_cost,
            "total_payoff": df["total_payoff"].sum(),
            "total_pnl": total_pnl,
            "portfolio_return_pct": total_pnl / total_cost if total_cost != 0 else np.nan,
            "avg_return_pct": df["return_pct"].mean(),
            "weighted_return_pct": np.average(
                df["return_pct"],
                weights=df["total_cost"]
            ) if total_cost != 0 else np.nan,
            "worst_product_pnl": df["pnl"].min(),
            "best_product_pnl": df["pnl"].max(),
            "barrier_breached_count": df["barrier_breached"].sum(),
            "near_barrier_count": (df["distance_to_barrier"] <= 0.05).sum()
        }])

    # =========================
    # 3. UNDERLYING LOOKTHROUGH
    # =========================
    def underlying_lookthrough(self) -> pd.DataFrame:

        """
        Provides underlying-level exposure view across all products.

        Each underlying inherits the full product exposure.
        This reflects payoff dependency (worst-of structure),
        not diversification-adjusted allocation.
        """
            
        rows = []

        for _, row in self.portfolio.iterrows():
            rc = ReverseConvertible(row)

            product_cost = rc.total_cost()
            current_perfs = rc.current_performances()
            final_perfs = rc.performances()
            barrier_distances = rc.current_barrier_distances()

            for i, underlying in enumerate(row["underlyings"]):
                rows.append({
                    "product_id": row["product_id"],
                    "underlying": underlying,
                    "isin": row["underlying_isins"][i],
                    "allocated_cost": product_cost,
                    "current_performance": current_perfs[i] - 1,
                    "scenario_performance": final_perfs[i] - 1,
                    "distance_to_barrier": barrier_distances[i],
                    "is_worst_underlying": underlying == rc.worst_underlying()
                })

        df = pd.DataFrame(rows)

        summary = (
            df.groupby(["underlying", "isin"], as_index=False)
            .agg(
                n_products=("product_id", "nunique"),
                allocated_cost=("allocated_cost", "sum"),
                avg_current_performance=("current_performance", "mean"),
                avg_scenario_performance=("scenario_performance", "mean"),
                min_distance_to_barrier=("distance_to_barrier", "min"),
                worst_of_count=("is_worst_underlying", "sum")
            )
            .sort_values("allocated_cost", ascending=False)
            .reset_index(drop=True)
        )

        total_alloc = summary["allocated_cost"].sum()
        summary["weight"] = (
            summary["allocated_cost"] / total_alloc if total_alloc != 0 else np.nan
        )

        return summary

    # =========================
    # 4. BARRIER WATCHLIST
    # =========================
    def barrier_watchlist(self, threshold: float = 0.05) -> pd.DataFrame:
        df = self._require_product_df().copy()
        df["near_barrier_flag"] = df["distance_to_barrier"] <= threshold

        return df.sort_values(
            ["near_barrier_flag", "distance_to_barrier", "pnl"],
            ascending=[False, True, True]
        ).reset_index(drop=True)

    # =========================
    # 5. MATURITY PROFILE
    # =========================
    def maturity_profile(self, today=None) -> pd.DataFrame:
        df = self._require_product_df().copy()

        if today is None:
            today = datetime.today()

        df["maturity_date_dt"] = pd.to_datetime(df["maturity_date"])
        df["days_to_maturity"] = (df["maturity_date_dt"] - pd.Timestamp(today)).dt.days

        bins = [-np.inf, 30, 90, 180, 365, 730, np.inf]
        labels = ["<=1M", "1-3M", "3-6M", "6-12M", "1-2Y", ">2Y"]

        df["maturity_bucket"] = pd.cut(df["days_to_maturity"], bins=bins, labels=labels)

        return (
            df.groupby("maturity_bucket", as_index=False, observed=False)
            .agg(
                n_products=("product_id", "count"),
                total_cost=("total_cost", "sum"),
                total_payoff=("total_payoff", "sum"),
                total_pnl=("pnl", "sum")
            )
        )

    # =========================
    # 6. SCENARIO ATTRIBUTION
    # =========================
    def scenario_attribution(self) -> pd.DataFrame:
        df = self._require_product_df().copy()

        total_pnl = df["pnl"].sum()

        df["pnl_contribution_pct"] = (
            df["pnl"] / total_pnl if total_pnl != 0 else np.nan
        )

        df["cost_weight"] = (
            df["total_cost"] / df["total_cost"].sum()
            if df["total_cost"].sum() != 0 else np.nan
        )

        return df[
            [
                "product_id",
                "product_type",
                "underlyings",
                "worst_underlying",
                "barrier_breached",
                "total_cost",
                "pnl",
                "pnl_contribution_pct",
                "return_pct",
                "distance_to_barrier"
            ]
        ].sort_values("pnl", ascending=True).reset_index(drop=True)

    # =========================
    # 7. RUN ALL
    # =========================
    def run_all(self) -> dict:
        self.build_product_analytics()

        return {
            "product_analytics": self.product_df,
            "portfolio_summary": self.portfolio_summary(),
            "underlying_lookthrough": self.underlying_lookthrough(),
            "barrier_watchlist": self.barrier_watchlist(),
            "maturity_profile": self.maturity_profile(),
            "scenario_attribution": self.scenario_attribution()
        }

    # =========================
    # INTERNAL HELPER
    # =========================
    def _require_product_df(self):
        if self.product_df is None:
            raise ValueError("Run build_product_analytics() first.")
        return self.product_df
    
    def total_portfolio_metrics(self) -> dict:
        df = self._require_product_df()

        total_notional = (df["notional"] * df["position_units"]).sum()
        total_cost = df["total_cost"].sum()
        total_payoff = df["total_payoff"].sum()
        total_pnl = df["pnl"].sum()

        # portfolio return
        portfolio_return = total_pnl / total_cost if total_cost != 0 else np.nan

        # weighted return p.a.
        weighted_return_pa = np.average(
            df["return_pa"],
            weights=df["total_cost"]
        ) if total_cost != 0 else np.nan

        return {
            "total_products": len(df),
            "total_notional": total_notional,
            "total_cost": total_cost,
            "total_payoff": total_payoff,
            "total_pnl": total_pnl,
            "portfolio_return_pct": portfolio_return,
            "portfolio_return_pa": weighted_return_pa
        }
    
    def total_portfolio_table(self) -> pd.DataFrame:
        return pd.DataFrame([self.total_portfolio_metrics()])

In [181]:
analytics = PortfolioAnalytics(portfolio)

analytics.build_product_analytics()

analytics.total_portfolio_table()

,total_products,total_notional,total_cost,total_payoff,total_pnl,portfolio_return_pct,portfolio_return_pa
0,3,25000,24900.0,26305.714608,1405.714608,0.056454,0.055561


In [113]:
beta_table = pd.DataFrame({
    "isin": [
        "CH0432492467",
        "CH0012221716",
        "CH0012214059",
        "CH0012005267",
        "CH0012032048"
    ],
    "beta": [0.85, 1.10, 0.95, 0.80, 1.05]
})
scenarios = {
    "down_5": -5,
    "down_10": -10,
    "crash": -20,
    "up_10": 10
}

beta_map = dict(zip(beta_table["isin"], beta_table["beta"]))
beta_map

{'CH0432492467': 0.85,
 'CH0012221716': 1.1,
 'CH0012214059': 0.95,
 'CH0012005267': 0.8,
 'CH0012032048': 1.05}

In [167]:
engine = ScenarioEngine(portfolio, beta_map, scenarios)

In [168]:

df = engine.run(-10)
df["portfolio_summary"]

,market_shock,n_products,total_cost,total_payoff,total_pnl,portfolio_return_pct
0,-10,3,24900.0,23835.092657,-1064.907343,-0.042767


In [169]:
portfolio.iloc[1]["underlying_isins"]

['CH0012221716', 'CH0012214059', 'CH0012005267', 'CH0012032048']